### Importations

In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold

from utils import evaluate_feature_selection_cv, evaluate_sfs, rw_rmse
import numpy as np

### Dataset

In [2]:
df = pd.read_csv('data_student/data_labeled/X_train.csv')
df2 = pd.read_csv('data_student/data_labeled/X_test.csv')
df3 = pd.read_csv('data_student/data_labeled/y_train.csv', header=None)
df4 = pd.read_csv('data_student/data_labeled/y_test.csv', header=None)

X_learn = df.drop(['img_filename'], axis=1)
X_test = df2.drop(['img_filename'], axis=1)
y_learn = df3
y_test = df4

X_learn.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,profession,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight
0,161,104.55,2.53,116.47,16.55,7.80,3.98,services,high,very_high,very_low,23.69,113.28
1,106,99.67,2.43,104.45,14.19,7.32,4.26,services,low,high,low,21.89,85.26
2,126,119.53,2.61,145.95,15.30,8.01,4.13,craftsmanship,low,moderate,very_high,28.00,122.44
3,179,122.99,2.22,112.76,16.01,7.81,4.65,food_production,high,very_high,high,21.05,122.17
4,140,105.63,2.47,112.13,13.92,7.23,3.92,craftsmanship,low,moderate,moderate,36.38,102.84
5,93,134.50,2.60,141.16,10.70,7.41,4.63,services,moderate,very_high,very_high,22.87,134.32
6,100,150.34,2.31,145.67,14.01,7.47,4.76,manufacturing,low,very_high,very_low,31.40,123.93
7,138,102.98,2.30,108.00,16.29,8.03,4.39,services,very_low,high,very_high,25.66,130.74
8,154,120.88,2.49,110.70,13.16,7.70,3.40,services,low,moderate,very_high,30.50,144.19
9,102,107.54,2.34,111.38,13.93,7.41,4.43,food_production,moderate,high,high,39.33,102.95


In [3]:
y_test.describe()

,0
count,500.000000
mean,0.084980
std,0.081471
min,0.000000
25%,0.030000
50%,0.060000
75%,0.110000
max,0.490000


### Missing values

In [4]:
X_learn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1000 non-null   int64  
 1   blood_pressure     1000 non-null   float64
 2   calcium            1000 non-null   float64
 3   cholesterol        1000 non-null   float64
 4   hemoglobin         1000 non-null   float64
 5   height             1000 non-null   float64
 6   potassium          1000 non-null   float64
 7   profession         1000 non-null   object 
 8   sarsaparilla       1000 non-null   object 
 9   smurfberry_liquor  1000 non-null   object 
 10  smurfin_donuts     1000 non-null   object 
 11  vitamin_D          1000 non-null   float64
 12  weight             1000 non-null   float64
dtypes: float64(8), int64(1), object(4)
memory usage: 101.7+ KB


### Categorical variables

- Strategy A: One-Hot for *Profession*, Ordinal for the rest
- Strategy B: One-Hot for all four variables

In [5]:
df_combined = X_learn.copy()
df_combined["target"] = y_learn

cat_cols = [
    "profession",
    "sarsaparilla",
    "smurfberry_liquor",
    "smurfin_donuts"
]

for col in cat_cols:
    display(
        df_combined.groupby(col)["target"].agg(
            ["count", "mean", "median"]
        )
    )

,count,mean,median
profession,,,
administration_and_governance,94,0.112979,0.08
craftsmanship,92,0.077717,0.05
food_production,272,0.071985,0.06
manufacturing,179,0.076145,0.06
resource_extraction,171,0.062164,0.05
services,192,0.099219,0.07


,count,mean,median
sarsaparilla,,,
high,149,0.057919,0.04
low,350,0.090543,0.07
moderate,167,0.077605,0.06
very_high,118,0.046864,0.03
very_low,216,0.101157,0.08


,count,mean,median
smurfberry_liquor,,,
high,224,0.090714,0.06
low,141,0.072128,0.05
moderate,378,0.069339,0.05
very_high,165,0.107818,0.09
very_low,92,0.067065,0.04


,count,mean,median
smurfin_donuts,,,
high,221,0.087511,0.07
low,202,0.060050,0.04
moderate,220,0.064136,0.05
very_high,187,0.142086,0.12
very_low,170,0.050059,0.04


In [6]:
X_learn_A = X_learn.copy()
X_test_A = X_test.copy()

consumption_map = {
    "very_low": 0,
    "low": 1,
    "moderate": 2,
    "high": 3,
    "very_high": 4
}

ordinal_cols = [
    "sarsaparilla",
    "smurfberry_liquor",
    "smurfin_donuts"
]

for col in ordinal_cols:
    X_learn_A[col] = X_learn_A[col].replace(consumption_map)
    X_test_A[col] = X_test_A[col].replace(consumption_map)

X_learn_A = pd.get_dummies(X_learn_A, columns=["profession"], dtype=int)
X_test_A = pd.get_dummies(X_test_A, columns=["profession"], dtype=int)

X_test_A = X_test_A.reindex(columns=X_learn_A.columns, fill_value=0)

X_learn_A.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight,profession_administration_and_governance,profession_craftsmanship,profession_food_production,profession_manufacturing,profession_resource_extraction,profession_services
0,161,104.55,2.53,116.47,16.55,7.80,3.98,3,4,0,23.69,113.28,0,0,0,0,0,1
1,106,99.67,2.43,104.45,14.19,7.32,4.26,1,3,1,21.89,85.26,0,0,0,0,0,1
2,126,119.53,2.61,145.95,15.30,8.01,4.13,1,2,4,28.00,122.44,0,1,0,0,0,0
3,179,122.99,2.22,112.76,16.01,7.81,4.65,3,4,3,21.05,122.17,0,0,1,0,0,0
4,140,105.63,2.47,112.13,13.92,7.23,3.92,1,2,2,36.38,102.84,0,1,0,0,0,0
5,93,134.50,2.60,141.16,10.70,7.41,4.63,2,4,4,22.87,134.32,0,0,0,0,0,1
6,100,150.34,2.31,145.67,14.01,7.47,4.76,1,4,0,31.40,123.93,0,0,0,1,0,0
7,138,102.98,2.30,108.00,16.29,8.03,4.39,0,3,4,25.66,130.74,0,0,0,0,0,1
8,154,120.88,2.49,110.70,13.16,7.70,3.40,1,2,4,30.50,144.19,0,0,0,0,0,1
9,102,107.54,2.34,111.38,13.93,7.41,4.43,2,3,3,39.33,102.95,0,0,1,0,0,0


In [7]:
X_learn_B = X_learn.copy()
X_test_B = X_test.copy()

X_learn_B = pd.get_dummies(X_learn_B, columns=cat_cols, dtype=int)
X_test_B = pd.get_dummies(X_test_B, columns=cat_cols, dtype=int)

X_test_B = X_test_B.reindex(columns=X_learn_B.columns, fill_value=0)

X_learn_B.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,vitamin_D,weight,profession_administration_and_governance,...,smurfberry_liquor_high,smurfberry_liquor_low,smurfberry_liquor_moderate,smurfberry_liquor_very_high,smurfberry_liquor_very_low,smurfin_donuts_high,smurfin_donuts_low,smurfin_donuts_moderate,smurfin_donuts_very_high,smurfin_donuts_very_low
0,161,104.55,2.53,116.47,16.55,7.80,3.98,23.69,113.28,0,...,0,0,0,1,0,0,0,0,0,1
1,106,99.67,2.43,104.45,14.19,7.32,4.26,21.89,85.26,0,...,1,0,0,0,0,0,1,0,0,0
2,126,119.53,2.61,145.95,15.30,8.01,4.13,28.00,122.44,0,...,0,0,1,0,0,0,0,0,1,0
3,179,122.99,2.22,112.76,16.01,7.81,4.65,21.05,122.17,0,...,0,0,0,1,0,1,0,0,0,0
4,140,105.63,2.47,112.13,13.92,7.23,3.92,36.38,102.84,0,...,0,0,1,0,0,0,0,1,0,0
5,93,134.50,2.60,141.16,10.70,7.41,4.63,22.87,134.32,0,...,0,0,0,1,0,0,0,0,1,0
6,100,150.34,2.31,145.67,14.01,7.47,4.76,31.40,123.93,0,...,0,0,0,1,0,0,0,0,0,1
7,138,102.98,2.30,108.00,16.29,8.03,4.39,25.66,130.74,0,...,1,0,0,0,0,0,0,0,1,0
8,154,120.88,2.49,110.70,13.16,7.70,3.40,30.50,144.19,0,...,0,0,1,0,0,0,0,0,1,0
9,102,107.54,2.34,111.38,13.93,7.41,4.43,39.33,102.95,0,...,1,0,0,0,0,1,0,0,0,0


### Outliers

In [8]:
X_learn.describe(percentiles=[0.05,0.95])

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,vitamin_D,weight
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,119.885000,109.046440,2.454470,112.437600,14.222370,7.580640,4.286010,29.757290,105.459500
std,29.433078,11.131466,0.207102,23.076467,1.923225,0.403022,0.484628,5.045444,15.830449
min,26.000000,88.030000,1.780000,39.370000,8.020000,6.360000,2.780000,11.390000,60.130000
5%,73.000000,95.397500,2.119500,72.619000,11.098500,6.930000,3.449500,21.429500,81.018500
50%,119.500000,106.580000,2.450000,111.770000,14.215000,7.570000,4.270000,29.955000,104.175000
95%,170.050000,129.527000,2.810000,149.992500,17.260000,8.210000,5.090000,38.010000,132.830000
max,180.000000,163.620000,3.110000,189.060000,20.040000,8.930000,5.570000,48.610000,151.350000


### Standardization

In [9]:
model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

### Feature Selection

In [10]:
scores_A_mean = evaluate_feature_selection_cv(X_learn_A, y_learn, model, X_learn_A.columns, True)
print("\n")
scores_B_mean = evaluate_feature_selection_cv(X_learn_B, y_learn, model, X_learn_B.columns, True)


Selected features:
- age
- blood_pressure
- calcium
- cholesterol
- hemoglobin
- height
- potassium
- sarsaparilla
- smurfberry_liquor
- smurfin_donuts
- vitamin_D
- weight
- profession_administration_and_governance
- profession_craftsmanship
- profession_food_production
- profession_manufacturing
- profession_resource_extraction
- profession_services
----------------------------------
RW-RMSE per fold: [0.05720825 0.07047008 0.05772055 0.06048355 0.06582845]
Mean RW-RMSE: 0.06234


Selected features:
- age
- blood_pressure
- calcium
- cholesterol
- hemoglobin
- height
- potassium
- vitamin_D
- weight
- profession_administration_and_governance
- profession_craftsmanship
- profession_food_production
- profession_manufacturing
- profession_resource_extraction
- profession_services
- sarsaparilla_high
- sarsaparilla_low
- sarsaparilla_moderate
- sarsaparilla_very_high
- sarsaparilla_very_low
- smurfberry_liquor_high
- smurfberry_liquor_low
- smurfberry_liquor_moderate
- smurfberry_liquor_

In [11]:
features = ["age", "blood_pressure", "cholesterol"]

score = evaluate_feature_selection_cv(
    X_learn_A,
    y_learn,
    model,
    features,
    True
)

Selected features:
- age
- blood_pressure
- cholesterol
----------------------------------
RW-RMSE per fold: [0.06450208 0.07765079 0.06430155 0.06268628 0.0691129 ]
Mean RW-RMSE: 0.06765


In [12]:
class ClippedLinearRegression(LinearRegression):

    def predict(self, X):
        y_pred = super().predict(X)
        return np.clip(y_pred, 0, 1)

In [13]:
model = make_pipeline(
        StandardScaler(),
        ClippedLinearRegression()
    )

score_A_f, features_A_f, pipeline_A_f = evaluate_sfs(
    X_learn_A, y_learn, model, direction="forward"
)

score_A_b, features_A_b, pipeline_A_b = evaluate_sfs(
    X_learn_A, y_learn, model, direction="backward"
)

score_B_f, features_B_f, pipeline_B_f = evaluate_sfs(
    X_learn_B, y_learn, model, direction="forward"
)

score_B_b, features_B_b, pipeline_B_b = evaluate_sfs(
    X_learn_B, y_learn, model, direction="backward"
)

Direction: forward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.05796634 0.07027782 0.05802684 0.06054796 0.06827819]
Mean RW-RMSE: 0.06302
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.05796634 0.07027782 0.05802684 0.06054796 0.06750104]
Mean RW-RMSE: 0.06286
----------------------------------
Direction: forward
Selected features: ['age', 'blood_pressure', 'cholesterol', 'hemoglobin', 'height', 'weight']
RW-RMSE per fold: [0.05854074 0.07123174 0.0584089  0.06153351 0.06682614]
Mean RW-RMSE: 0.06331
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'cholesterol', 'hemoglobin', 'height', 'weight']
RW-RMSE per fold: [0.05854074 0.07122802 0.0584089  0.06153351 0.06682614]
Mean RW-RMSE: 0.06331
----------------------------------


In [14]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression

class WeightedLinearRegression(RegressorMixin, BaseEstimator):

    def fit(self, X, y):
        y = np.asarray(y).ravel()

        weights = 1 + y / 0.1

        self.model_ = LinearRegression()
        self.model_.fit(X, y, sample_weight=weights)

        return self

    def predict(self, X):
        predictions = self.model_.predict(X)
        return np.clip(predictions, 0, 1)

In [15]:
model_weighted = make_pipeline(
    StandardScaler(),
    WeightedLinearRegression()
)

score_A_final_forward, features_A_final_forward, pipeline_A_final_forward = evaluate_sfs(
    X_learn_A, y_learn, model_weighted, direction="forward"
)

score_final, features_final, pipeline_weighted_final = evaluate_sfs(
    X_learn_A, y_learn, model_weighted, direction="backward"
)

score_B_f, features_B_f, pipeline_B_f = evaluate_sfs(
    X_learn_B, y_learn, model_weighted, direction="forward"
)

score_B_b, features_B_b, pipeline_B_b = evaluate_sfs(
    X_learn_B, y_learn, model_weighted, direction="backward"
)

Direction: forward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.0570037  0.06556427 0.05447229 0.06264372 0.06085733]
Mean RW-RMSE: 0.06011
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.0570037  0.06556427 0.05447229 0.06264372 0.06124446]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: forward
Selected features: ['age', 'blood_pressure', 'cholesterol', 'hemoglobin', 'height', 'weight']
RW-RMSE per fold: [0.05793106 0.06657956 0.05463558 0.06331396 0.06019682]
Mean RW-RMSE: 0.06053
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'cholesterol', 'hemoglobin', 'height', 'weight']
RW-RMSE per fold: [0.058676   0.06657956 0.05463558 0.06331396 0.06019682]
Mean RW-RMSE: 0.06068
----------------------------------


In [16]:
import numpy as np

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge, Lasso

class WeightedRegression(RegressorMixin, BaseEstimator):

    def __init__(self, model_type="ridge", alpha=1.0):
        self.model_type = model_type
        self.alpha = alpha

    def fit(self, X, y):
        y = np.asarray(y).ravel()

        weights = 1 + y / 0.1

        if self.model_type == "ridge":
            self.model_ = Ridge(alpha=self.alpha)

        elif self.model_type == "lasso":
            self.model_ = Lasso(
                alpha=self.alpha,
                max_iter=20000
            )
        else:
            raise ValueError("Unknown model type")

        self.model_.fit(X, y, sample_weight=weights)

        return self

    def predict(self, X):
        predictions = self.model_.predict(X)
        return np.clip(predictions, 0, 1)

In [17]:
ridge_model = make_pipeline(
    StandardScaler(),
    WeightedRegression(model_type="ridge", alpha=1.0)
)

lasso_model = make_pipeline(
    StandardScaler(),
    WeightedRegression(model_type="lasso", alpha=0.001)
)

In [18]:
ridge_alphas = [0.01, 0.1, 1, 10, 100]

results_ridge = []

for alpha in ridge_alphas:

    model = make_pipeline(
        StandardScaler(),
        WeightedRegression(model_type="ridge", alpha=alpha)
    )

    score, features, pipeline = evaluate_sfs(
        X_learn_A,
        y_learn,
        model=model,
        direction="backward",
        n_features=6
    )

    results_ridge.append({
        "alpha": alpha,
        "RW-RMSE": score,
        "features": features
    })

Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.05700367 0.06556433 0.0544723  0.06264365 0.06124452]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.05700333 0.06556483 0.05447236 0.06264307 0.0612451 ]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.057      0.06556987 0.05447299 0.06263728 0.0612508 ]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.0569676  0.0656207  0.05447994 0.06257986 0.06130828]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: b

In [19]:
df_ridge = pd.DataFrame(results_ridge)

display(df_ridge.sort_values("RW-RMSE"))

,alpha,RW-RMSE,features
0,0.01,0.060186,"[age, blood_pressure, hemoglobin, height, sars..."
1,0.10,0.060186,"[age, blood_pressure, hemoglobin, height, sars..."
2,1.00,0.060186,"[age, blood_pressure, hemoglobin, height, sars..."
3,10.00,0.060191,"[age, blood_pressure, hemoglobin, height, sars..."
4,100.00,0.060294,"[age, blood_pressure, hemoglobin, height, sars..."


In [20]:
lasso_alphas = [0.00001, 0.0001, 0.001, 0.01]

results_lasso = []

for alpha in lasso_alphas:

    model = make_pipeline(
        StandardScaler(),
        WeightedRegression(model_type="lasso", alpha=alpha)
    )

    score, features, pipeline = evaluate_sfs(
        X_learn_A,
        y_learn,
        model=model,
        direction="backward",
        n_features=6
    )

    results_lasso.append({
        "alpha": alpha,
        "RW-RMSE": score,
        "features": features
    })

Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.05700509 0.06556892 0.05447171 0.06263933 0.06124555]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.05701729 0.06561116 0.05446709 0.06260027 0.0612557 ]
Mean RW-RMSE: 0.06019
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.0571733  0.06606683 0.0544621  0.0622307  0.06137835]
Mean RW-RMSE: 0.06026
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'cholesterol', 'hemoglobin', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.06126969 0.07281221 0.05804817 0.06136585 0.06485062]
Mean RW-RMSE: 0.06367
----------------------------------


In [21]:
df_lasso = pd.DataFrame(results_lasso)

display(df_lasso.sort_values("RW-RMSE"))

,alpha,RW-RMSE,features
0,0.00001,0.060186,"[age, blood_pressure, hemoglobin, height, sars..."
1,0.00010,0.060190,"[age, blood_pressure, hemoglobin, height, sars..."
2,0.00100,0.060262,"[age, blood_pressure, hemoglobin, height, sars..."
3,0.01000,0.063669,"[age, blood_pressure, cholesterol, hemoglobin,..."


In [22]:
n_features_values = [2, 3, 4, 5, 6]

results = []

for n in n_features_values:
    score, features, pipeline = evaluate_sfs(
        X_learn_A,
        y_learn,
        model=model_weighted,
        direction="backward",
        n_features=n
    )

    results.append({
        "n_features": n,
        "RW-RMSE": score,
        "features": features
    })

df_results = pd.DataFrame(results)
display(df_results.sort_values("RW-RMSE"))

Direction: backward
Selected features: ['blood_pressure', 'hemoglobin']
RW-RMSE per fold: [0.06357672 0.07184644 0.0593024  0.06518725 0.06497145]
Mean RW-RMSE: 0.06498
----------------------------------
Direction: backward
Selected features: ['blood_pressure', 'hemoglobin', 'weight']
RW-RMSE per fold: [0.06269242 0.07038005 0.05692863 0.06631344 0.06474809]
Mean RW-RMSE: 0.06421
----------------------------------
Direction: backward
Selected features: ['blood_pressure', 'hemoglobin', 'height', 'weight']
RW-RMSE per fold: [0.060521   0.06995246 0.0546995  0.06427444 0.06260197]
Mean RW-RMSE: 0.06241
----------------------------------
Direction: backward
Selected features: ['blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-RMSE per fold: [0.0590518  0.06744689 0.05546199 0.06394954 0.06089555]
Mean RW-RMSE: 0.06136
----------------------------------
Direction: backward
Selected features: ['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']
RW-

,n_features,RW-RMSE,features
4,6,0.060186,"[age, blood_pressure, hemoglobin, height, sars..."
3,5,0.061361,"[blood_pressure, hemoglobin, height, sarsapari..."
2,4,0.062410,"[blood_pressure, hemoglobin, height, weight]"
1,3,0.064213,"[blood_pressure, hemoglobin, weight]"
0,2,0.064977,"[blood_pressure, hemoglobin]"


### RW-RMSE on Test set


In [23]:
y_pred_test_final_forward = pipeline_A_final_forward.predict(X_test_A)

print(
    "Test RW-RMSE:",
    rw_rmse(y_test, y_pred_test_final_forward)
)

print(features_A_final_forward)

Test RW-RMSE: 0.06450009831758648
['age', 'blood_pressure', 'hemoglobin', 'height', 'sarsaparilla', 'weight']


### Prediction on unlabeled set

In [24]:
X_unlabeled = pd.read_csv(
    "data_student/data_unlabeled/X.csv"
)

print(X_unlabeled.shape)
X_unlabeled.head()

(500, 14)


,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,profession,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight,img_filename
0,115,101.06,2.23,144.81,14.37,8.08,4.66,resource_extraction,moderate,moderate,moderate,15.00,94.11,heart_0.png
1,74,131.72,2.59,151.69,18.65,7.34,4.02,food_production,very_low,moderate,very_high,30.56,117.81,heart_1.png
2,130,98.93,2.47,87.04,14.05,7.12,4.28,administration_and_governance,moderate,low,moderate,34.07,85.47,heart_2.png
3,173,123.06,2.71,141.25,13.18,7.69,4.53,resource_extraction,very_low,moderate,very_high,30.76,88.49,heart_3.png
4,110,100.69,2.36,79.57,16.08,7.82,3.81,food_production,low,high,very_low,32.88,77.13,heart_4.png


In [25]:
X_full = pd.concat([X_learn, X_test], ignore_index=True)
y_full = pd.concat([y_learn, y_test], ignore_index=True)

print(X_full.shape, y_full.shape)
X_full.head(10)

(1500, 13) (1500, 1)


,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,profession,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight
0,161,104.55,2.53,116.47,16.55,7.80,3.98,services,high,very_high,very_low,23.69,113.28
1,106,99.67,2.43,104.45,14.19,7.32,4.26,services,low,high,low,21.89,85.26
2,126,119.53,2.61,145.95,15.30,8.01,4.13,craftsmanship,low,moderate,very_high,28.00,122.44
3,179,122.99,2.22,112.76,16.01,7.81,4.65,food_production,high,very_high,high,21.05,122.17
4,140,105.63,2.47,112.13,13.92,7.23,3.92,craftsmanship,low,moderate,moderate,36.38,102.84
5,93,134.50,2.60,141.16,10.70,7.41,4.63,services,moderate,very_high,very_high,22.87,134.32
6,100,150.34,2.31,145.67,14.01,7.47,4.76,manufacturing,low,very_high,very_low,31.40,123.93
7,138,102.98,2.30,108.00,16.29,8.03,4.39,services,very_low,high,very_high,25.66,130.74
8,154,120.88,2.49,110.70,13.16,7.70,3.40,services,low,moderate,very_high,30.50,144.19
9,102,107.54,2.34,111.38,13.93,7.41,4.43,food_production,moderate,high,high,39.33,102.95


In [26]:
X_full_A = X_full.copy()
X_unlabeled_A = X_unlabeled.drop(columns=["img_filename"]).copy()

for col in ordinal_cols:
    X_full_A[col] = X_full_A[col].replace(consumption_map)
    X_unlabeled_A[col] = X_unlabeled_A[col].replace(consumption_map)

X_full_A = pd.get_dummies(X_full_A, columns=["profession"], dtype=int)
X_unlabeled_A = pd.get_dummies(X_unlabeled_A, columns=["profession"], dtype=int)

X_unlabeled_A = X_unlabeled_A.reindex(columns=X_full_A.columns, fill_value=0)

X_unlabeled_A.head(10)

,age,blood_pressure,calcium,cholesterol,hemoglobin,height,potassium,sarsaparilla,smurfberry_liquor,smurfin_donuts,vitamin_D,weight,profession_administration_and_governance,profession_craftsmanship,profession_food_production,profession_manufacturing,profession_resource_extraction,profession_services
0,115,101.06,2.23,144.81,14.37,8.08,4.66,2,2,2,15.00,94.11,0,0,0,0,1,0
1,74,131.72,2.59,151.69,18.65,7.34,4.02,0,2,4,30.56,117.81,0,0,1,0,0,0
2,130,98.93,2.47,87.04,14.05,7.12,4.28,2,1,2,34.07,85.47,1,0,0,0,0,0
3,173,123.06,2.71,141.25,13.18,7.69,4.53,0,2,4,30.76,88.49,0,0,0,0,1,0
4,110,100.69,2.36,79.57,16.08,7.82,3.81,1,3,0,32.88,77.13,0,0,1,0,0,0
5,103,100.99,2.64,112.29,13.25,7.55,3.43,2,2,0,28.31,92.55,0,0,0,1,0,0
6,103,99.87,2.52,134.13,12.49,7.85,4.97,2,2,0,34.84,85.68,1,0,0,0,0,0
7,97,96.46,2.69,128.07,14.94,7.76,4.74,0,1,1,23.89,91.20,0,0,0,1,0,0
8,70,108.54,2.64,120.28,16.41,7.36,3.93,4,4,2,20.82,89.43,0,0,0,1,0,0
9,178,106.88,2.36,115.14,14.96,7.80,4.41,2,1,2,23.29,109.44,0,0,1,0,0,0


In [27]:
model_final = make_pipeline(
    StandardScaler(),
    WeightedRegression()
)

model_final.fit(
    X_full_A[features_A_final_forward],
    y_full
)

y_pred_unlabeled = model_final.predict(
    X_unlabeled_A[features_A_final_forward]
)

pd.Series(y_pred_unlabeled).to_csv(
    "y_pred.csv",
    header=False,
    index=False
)